In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

# Notebook lives in rag/build-samples/, corpus in <root>/corpus/
ROOT = Path.cwd().resolve()
while not (ROOT / "corpus" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW_DIR = ROOT / "corpus" / "raw"
OUT_DIR = ROOT / "corpus" / "clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("raw:", RAW_DIR, "\nout:", OUT_DIR)

In [ ]:
FRONT_RE = re.compile(r"\A---\s*\n(.*?)\n---\s*\n", re.S)
TS_RE = re.compile(r"^\[(\d{1,2}:\d{2}:\d{2})\]\s*(.*)$")


def parse_front_matter(raw: str):
    m = FRONT_RE.match(raw)
    if not m:
        raise ValueError("front matter not found")
    meta = {}
    for line in m.group(1).splitlines():
        key, _, value = line.partition(":")
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] == '"':
            value = value[1:-1]
        meta[key.strip()] = value
    return meta, raw[m.end():]


def parse_doc(path: Path) -> dict:
    raw = path.read_text(encoding="utf-8-sig").replace("\r\n", "\n")
    meta, body = parse_front_matter(raw)

    entries: list[dict[str, str]] = []
    for line in body.split("\n"):
        if not line.strip():
            continue
        m = TS_RE.match(line)
        if m:
            entries.append({m.group(1): m.group(2).strip()})
        elif entries:  # continuation line without a timecode -> glue to previous entry
            (tc, txt), = entries[-1].items()
            entries[-1][tc] = f"{txt} {line.strip()}".strip()
        else:
            raise ValueError(f"{path.name}: text before first timecode: {line[:60]!r}")

    return {
        "title": meta["title"],
        "date": meta["date"],
        "youtubeId": meta["youtubeId"],
        "url": meta["url"],
        "duration": meta["duration"],
        "text": entries,
    }

In [ ]:
# ---- Cleaning: conservative, only disfluencies (no stop-word removal) ----
W = r"[\w'ʼ’]+"          # a word (apostrophes are part of Ukrainian words)
NOT_WORD = r"[\w'ʼ’-]"   # chars that must not touch a hesitation sound

# ASR sound tags: [сміх], [сопіння], [зітхання], ... and the literal "(null)"
TAG_RE = re.compile(r"\[[^\]]*\]|\(null\)", re.I)

# hesitation sounds: е, е-е, ее, а-а, аа, м, м-м, мм, хм ...
# NB: single "а" is a real conjunction -> only the doubled forms are removed
HES_RE = re.compile(
    rf"(?<!{NOT_WORD})(?:е(?:-?е)*|а(?:-а)+|аа+|м(?:-?м)*|хм+)(?!{NOT_WORD})", re.I
)

# speaker-turn marker from the captions
SPEAKER_RE = re.compile(r">>")

# discourse fillers (extend this list as needed)
FILLER_WORDS = ["ну", "от", "короче", "кароче", "коротше"]
FILLER_RE = re.compile(rf"(?<!{NOT_WORD})(?:{'|'.join(FILLER_WORDS)})(?!{NOT_WORD})", re.I)
# "там" is also a real place adverb ("там, в Лондоні"), so it is removed only when used
# as a filler, i.e. set off by commas / sentence start: ", там," | ". Там, ..." | "Там, ..."
TAM_RE = re.compile(r"(^|[.!?]\s+)там\s*,\s*|,\s*там\s*(?=,)", re.I)

# stutters: "ну воно, ну воно" (2-3 word phrase) and "да да" (single word)
PHRASE_REP_RE = re.compile(rf"(?<![\w'ʼ’])({W}(?:\s+{W}){{1,2}})(?:[,\s]+\1)+(?![\w'ʼ’])", re.I)
WORD_REP_RE = re.compile(rf"(?<![\w'ʼ’])({W})(?:[,\s]+\1)+(?![\w'ʼ’])", re.I)

STATS = Counter()


def _sub(pattern, repl, s, key):
    s, n = pattern.subn(repl, s)
    STATS[key] += n
    return s


def tidy(s: str) -> str:
    """Fix punctuation/spaces left behind after removals."""
    s = re.sub(r"\s+([,.;:!?…])", r"\1", s)        # "слово ," -> "слово,"
    s = re.sub(r"([,;:])(?:\s*[,;:])+", r"\1", s)   # ", ," -> ","
    s = re.sub(r",\s*([.!?…])", r"\1", s)           # ",." -> "."
    s = re.sub(r"([.!?…])\s*,", r"\1", s)           # ".," -> "." (hesitation removed after a full stop)
    s = re.sub(r"(?<!\.)\.\.(?!\.)", ".", s)        # "слово. ." (a sentence was removed) -> "."
    s = re.sub(r"^[\s,;:.]+", "", s)                # leading leftovers
    return re.sub(r"\s{2,}", " ", s).strip()


def clean_text(s: str) -> str:
    s = _sub(TAG_RE, " ", s, "sound tags")
    s = _sub(SPEAKER_RE, " ", s, "speaker markers (>>)")
    s = _sub(HES_RE, " ", s, "hesitations (е-е, а-а, м-м...)")
    s = _sub(FILLER_RE, " ", s, "fillers (ну, от, короче...)")
    s = tidy(s)
    s = _sub(TAM_RE, lambda m: m.group(1) or "", s, "filler 'там'")
    s = tidy(s)
    for _ in range(5):  # repeat until stable ("да да да да" -> "да да" -> "да")
        before = s
        s = _sub(PHRASE_REP_RE, r"\1", s, "repeated phrases")
        s = _sub(WORD_REP_RE, r"\1", s, "repeated words")
        if s == before:
            break
    return tidy(s)


def clean_entries(entries: list[dict[str, str]]) -> list[dict[str, str]]:
    out = []
    for e in entries:
        (tc, txt), = e.items()
        c = clean_text(txt)
        if re.search(r"\w", c):
            out.append({tc: c})
        else:
            STATS["dropped empty entries"] += 1
    return out

In [ ]:
docs = []
for path in sorted(RAW_DIR.glob("*.md")):
    doc = parse_doc(path)
    doc["text_clean"] = clean_entries(doc["text"])   # "text" stays raw
    (OUT_DIR / f"{path.stem}.json").write_text(
        json.dumps(doc, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    docs.append(doc)
print(f"{len(docs)} docs -> {OUT_DIR}")

In [ ]:
# ---- Sanity checks + before/after summary ----
def to_sec(tc: str) -> int:
    h, m, s = map(int, tc.split(":"))
    return h * 3600 + m * 60 + s

def n_words(entries):
    return sum(len(next(iter(e.values())).split()) for e in entries)

tot_raw = tot_clean = 0
for d in docs:
    for key in ("text", "text_clean"):
        secs = [to_sec(next(iter(e))) for e in d[key]]
        assert secs == sorted(secs), f"{d['youtubeId']}: {key} timecodes not monotonic"
        assert secs[-1] <= to_sec(d["duration"]), f"{d['youtubeId']}: {key} timecode beyond duration"
    r, c = n_words(d["text"]), n_words(d["text_clean"])
    tot_raw += r; tot_clean += c
    print(f"{d['date']} {d['youtubeId']:12} entries {len(d['text']):4} -> {len(d['text_clean']):4}   words {r:6} -> {c:6}")

print(f"\nTOTAL words {tot_raw} -> {tot_clean}  (-{100 * (1 - tot_clean / tot_raw):.1f}%)")
print("\nremoved occurrences by rule:")
for k, v in STATS.most_common():
    print(f"  {v:6}  {k}")

# leftovers that should not survive cleaning
left = Counter()
for d in docs:
    for e in d["text_clean"]:
        t = next(iter(e.values()))
        if "[" in t or "]" in t: left["brackets"] += 1
        if ">>" in t: left["speaker marker"] += 1
        if FILLER_RE.search(t): left["filler"] += 1
        if re.search(r"(?<![\w'ʼ’-])(?:е|м|аа+|а-а|е-е|м-м)(?![\w'ʼ’-])", t, re.I): left["hesitation"] += 1
        if re.search(r"[,;:]\s*[,;:]|[.!?…],|(?<!\.)\.\.(?!\.)", t): left["double punct"] += 1
print("leftovers:", dict(left) or "none")

In [ ]:
# ---- Examples: entries changed the most ----
rows = []
for d in docs:
    for e_raw in d["text"]:
        (tc, raw_t), = e_raw.items()
        clean_t = next((v for e in d["text_clean"] for k, v in e.items() if k == tc), "")
        rows.append((len(raw_t) - len(clean_t), d["youtubeId"], tc, raw_t, clean_t))
for diff, vid, tc, raw_t, clean_t in sorted(rows, reverse=True)[:5]:
    print(f"--- {vid} [{tc}]  -{diff} chars\nRAW  : {raw_t}\nCLEAN: {clean_t}\n")